# Image Preprocessing for the Pair Dataset

Two goals:

1. **Crop to the vehicle** with the YOLO detector trained in `notebooks/detectors/vehicle/best.pt` — this kills the background bias (the model was learning parking lots, not cars).
2. **Dedup cross-listings** with perceptual hashes — the same photo often appears in multiple listings (different `vehicle_id`), which leaks identity into negatives. We drop any image that shows up under more than one `vehicle_id`.

Output:
- `pairs/images_cropped/<sha1>.jpg` — cropped, jpeg-encoded survivors.
- `pairs/pairs_{train,val,test}_clean.csv` — pair CSVs keeping only rows whose **both** images survived.

After this, `TrainCosineEncoder.ipynb` only needs `DATA_ROOT` re-pointed and CSV filenames updated.

In [ ]:
%pip install -q ultralytics imagehash pillow tqdm pandas

In [ ]:
import os
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
import imagehash
from tqdm.auto import tqdm
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## Paths

In [ ]:
PAIRS_ROOT   = Path("pairs")               # produced by BuildPairsDataset.ipynb
SRC_IMG_DIR  = PAIRS_ROOT / "images"
DST_IMG_DIR  = PAIRS_ROOT / "images_cropped"
DST_IMG_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUTS = {
    "train": PAIRS_ROOT / "pairs_train.csv",
    "val":   PAIRS_ROOT / "pairs_val.csv",
    "test":  PAIRS_ROOT / "pairs_test.csv",
}
CSV_OUTPUTS = {k: PAIRS_ROOT / f"pairs_{k}_clean.csv" for k in CSV_INPUTS}

# Pretrained COCO YOLOv8 — much more reliable than our custom GTA-trained detector
# on real marketplace photos. ultralytics downloads it automatically on first use.
#  - yolov8n.pt (6MB)  — fast, good baseline
#  - yolov8s.pt (22MB) — better recall, ~2x slower
YOLO_WEIGHTS = "yolov8n.pt"

# COCO class IDs we treat as "vehicle"
# 2=car, 5=bus, 7=truck   (motorcycle=3 intentionally excluded)
VEHICLE_CLASS_IDS = [2, 5, 7]

# Tunables
CONF_THRESHOLD = 0.4     # minimum detection confidence (a touch lower for COCO than for our custom)
MIN_AREA_RATIO = 0.15    # vehicle bbox must cover >= 15% of the image
PAD_RATIO      = 0.10    # 10% padding around the bbox
BATCH_SIZE     = 32      # YOLO inference batch
OUT_JPEG_Q     = 92      # quality for re-saved cropped JPEGs
PHASH_SIZE     = 16      # phash dimensionality (16x16 = 256 bits) — strict matching

## Collect work list

Unique image paths across all three CSVs, plus a mapping `image -> set(vehicle_id)` for later dedup.

In [ ]:
dfs = {k: pd.read_csv(p) for k, p in CSV_INPUTS.items()}
for k, df in dfs.items():
    print(f"{k}: {len(df)} pairs")

img_to_vids: dict[str, set[int]] = defaultdict(set)
for df in dfs.values():
    for _, row in df.iterrows():
        img_to_vids[row["path_a"]].add(int(row["vehicle_id_a"]))
        img_to_vids[row["path_b"]].add(int(row["vehicle_id_b"]))

all_images = sorted(img_to_vids.keys())
print(f"Unique images to process: {len(all_images)}")

model = YOLO(YOLO_WEIGHTS)
print("Classes:", model.names)
print("Filtering to:", {i: model.names[i] for i in VEHICLE_CLASS_IDS})

In [ ]:
def expand_and_clip_box(x1, y1, x2, y2, W, H, pad_ratio):
    bw, bh = x2 - x1, y2 - y1
    px, py = bw * pad_ratio, bh * pad_ratio
    nx1 = max(0, int(round(x1 - px)))
    ny1 = max(0, int(round(y1 - py)))
    nx2 = min(W, int(round(x2 + px)))
    ny2 = min(H, int(round(y2 + py)))
    return nx1, ny1, nx2, ny2


def safe_open(abs_path: str) -> Image.Image | None:
    """Try to fully load the image with PIL. Returns None for unreadable / empty / corrupt files."""
    try:
        if not os.path.exists(abs_path) or os.path.getsize(abs_path) == 0:
            return None
        img = Image.open(abs_path)
        img.load()                 # force full decode now to catch truncation
        return img.convert("RGB")
    except Exception:
        return None


def _process_with_image(rel: str, img: Image.Image, det_result):
    """Common post-processing once we have a PIL image and a YOLO Results object."""
    W, H = img.size
    boxes = det_result.boxes
    if boxes is None or len(boxes) == 0:
        return (rel, None, None, "no_detection")

    xyxy = boxes.xyxy.detach().cpu().numpy()
    areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    best_idx = int(areas.argmax())
    x1, y1, x2, y2 = xyxy[best_idx].tolist()
    area_ratio = float(areas[best_idx]) / (W * H + 1e-9)

    if area_ratio < MIN_AREA_RATIO:
        return (rel, None, None, f"too_small:{area_ratio:.3f}")

    nx1, ny1, nx2, ny2 = expand_and_clip_box(x1, y1, x2, y2, W, H, PAD_RATIO)
    crop = img.crop((nx1, ny1, nx2, ny2))

    stem = Path(rel).stem
    dst_rel = f"images_cropped/{stem}.jpg"
    dst_abs = PAIRS_ROOT / dst_rel
    crop.save(dst_abs, format="JPEG", quality=OUT_JPEG_Q)

    ph = str(imagehash.phash(crop, hash_size=PHASH_SIZE))
    return (rel, dst_rel, ph, None)


def _predict(images: list[Image.Image]):
    """Run YOLO on a list of already-loaded PIL images, filtering to vehicle classes."""
    return model.predict(
        source=images,
        conf=CONF_THRESHOLD,
        classes=VEHICLE_CLASS_IDS,
        device=DEVICE,
        verbose=False,
    )


def process_batch(rel_paths: list[str]):
    """
    Robust batched processing: pre-validates each file with PIL, then tries a
    batched YOLO call. If the batched call raises (one bad file in OpenCV), falls
    back to per-image inference so a single bad sample doesn't kill the batch.
    """
    out = []
    loaded: list[tuple[str, Image.Image]] = []
    for rel in rel_paths:
        img = safe_open(str(PAIRS_ROOT / rel))
        if img is None:
            out.append((rel, None, None, "open_failed"))
        else:
            loaded.append((rel, img))

    if not loaded:
        return out

    try:
        results = _predict([img for _, img in loaded])
        for (rel, img), r in zip(loaded, results):
            out.append(_process_with_image(rel, img, r))
    except Exception:
        # Per-image fallback — isolate the bad sample
        for rel, img in loaded:
            try:
                r = _predict([img])[0]
                out.append(_process_with_image(rel, img, r))
            except Exception as e:
                out.append((rel, None, None, f"yolo_failed:{e.__class__.__name__}"))
    return out


records: dict[str, dict] = {}   # rel_input -> {"out": rel_output, "phash": str, "reason": str|None}
for i in tqdm(range(0, len(all_images), BATCH_SIZE), desc="crop"):
    batch = all_images[i:i + BATCH_SIZE]
    for rel, dst_rel, ph, reason in process_batch(batch):
        records[rel] = {"out": dst_rel, "phash": ph, "reason": reason}

n_ok    = sum(1 for v in records.values() if v["out"])
n_drop  = len(records) - n_ok
reasons = pd.Series([v["reason"] for v in records.values() if v["reason"]]).value_counts()
print(f"\nCropped OK: {n_ok} | Dropped: {n_drop}")
if not reasons.empty:
    print("\nDrop reasons:\n" + reasons.to_string())

## Crop pass

For each image:
1. Run YOLO.
2. Pick the **largest** bbox with `conf >= CONF_THRESHOLD`.
3. If bbox area / image area `< MIN_AREA_RATIO` → drop (likely interior shot, document, or distant car).
4. Pad bbox by `PAD_RATIO`, clip to image bounds, crop.
5. Save as JPEG to `images_cropped/`.
6. Compute `phash` on the **cropped** image (more meaningful than on the raw shot).

In [ ]:
def expand_and_clip_box(x1, y1, x2, y2, W, H, pad_ratio):
    bw, bh = x2 - x1, y2 - y1
    px, py = bw * pad_ratio, bh * pad_ratio
    nx1 = max(0, int(round(x1 - px)))
    ny1 = max(0, int(round(y1 - py)))
    nx2 = min(W, int(round(x2 + px)))
    ny2 = min(H, int(round(y2 + py)))
    return nx1, ny1, nx2, ny2


def process_batch(rel_paths: list[str]) -> list[tuple[str, str | None, str | None, str | None]]:
    """
    Returns list of tuples per input path:
        (rel_path, output_rel_path_or_None, phash_hex_or_None, reason_or_None)
    """
    abs_paths = [str(PAIRS_ROOT / p) for p in rel_paths]
    results = model.predict(
        source=abs_paths,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )
    out = []
    for rel, abs_p, r in zip(rel_paths, abs_paths, results):
        try:
            img = Image.open(abs_p).convert("RGB")
        except Exception as e:
            out.append((rel, None, None, f"open_failed:{e.__class__.__name__}"))
            continue

        W, H = img.size
        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            out.append((rel, None, None, "no_detection"))
            continue

        xyxy = boxes.xyxy.detach().cpu().numpy()
        areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
        best_idx = int(areas.argmax())
        x1, y1, x2, y2 = xyxy[best_idx].tolist()
        area_ratio = float(areas[best_idx]) / (W * H + 1e-9)

        if area_ratio < MIN_AREA_RATIO:
            out.append((rel, None, None, f"too_small:{area_ratio:.3f}"))
            continue

        nx1, ny1, nx2, ny2 = expand_and_clip_box(x1, y1, x2, y2, W, H, PAD_RATIO)
        crop = img.crop((nx1, ny1, nx2, ny2))

        # Save as jpeg; same stem so it's still recognisable
        stem = Path(rel).stem
        dst_rel = f"images_cropped/{stem}.jpg"
        dst_abs = PAIRS_ROOT / dst_rel
        crop.save(dst_abs, format="JPEG", quality=OUT_JPEG_Q)

        ph = str(imagehash.phash(crop, hash_size=PHASH_SIZE))
        out.append((rel, dst_rel, ph, None))
    return out


records: dict[str, dict] = {}   # rel_input -> {"out": rel_output, "phash": str, "reason": str|None}
for i in tqdm(range(0, len(all_images), BATCH_SIZE), desc="crop"):
    batch = all_images[i:i + BATCH_SIZE]
    for rel, dst_rel, ph, reason in process_batch(batch):
        records[rel] = {"out": dst_rel, "phash": ph, "reason": reason}

n_ok    = sum(1 for v in records.values() if v["out"])
n_drop  = len(records) - n_ok
reasons = pd.Series([v["reason"] for v in records.values() if v["reason"]]).value_counts()
print(f"\nCropped OK: {n_ok} | Dropped: {n_drop}")
if not reasons.empty:
    print("\nDrop reasons:\n" + reasons.to_string())

## Cross-listing dedup

Group surviving images by `phash`. If a hash group spans **more than one `vehicle_id`**, that's the same physical photo re-used across listings (or two near-identical photos of different cars — rare but possible) — drop the entire group.

If a group has multiple images **of the same vehicle**, we keep all of them — they are honest duplicates within one listing, harmless.

In [ ]:
phash_groups: dict[str, list[str]] = defaultdict(list)
for rel, info in records.items():
    if info["out"] is not None:
        phash_groups[info["phash"]].append(rel)

dropped_dups: set[str] = set()
n_cross_groups = 0
for ph, members in phash_groups.items():
    if len(members) < 2:
        continue
    vids: set[int] = set()
    for rel in members:
        vids |= img_to_vids.get(rel, set())
    if len(vids) > 1:
        n_cross_groups += 1
        dropped_dups.update(members)

print(f"Cross-vehicle phash groups: {n_cross_groups}")
print(f"Images dropped as cross-listings: {len(dropped_dups)}")

# Final survivor set: cropped AND not flagged as cross-listing
survivors = {
    rel: info["out"]
    for rel, info in records.items()
    if info["out"] is not None and rel not in dropped_dups
}
print(f"Final survivors: {len(survivors)} / {len(all_images)}")

## Rewrite CSVs

Keep only pairs where **both** images survived; rewrite paths to the cropped versions.

In [ ]:
for split, df in dfs.items():
    keep = []
    for _, row in df.iterrows():
        a = survivors.get(row["path_a"])
        b = survivors.get(row["path_b"])
        if not a or not b:
            continue
        keep.append({
            "path_a": a,
            "path_b": b,
            "label":  int(row["label"]),
            "vehicle_id_a": int(row["vehicle_id_a"]),
            "vehicle_id_b": int(row["vehicle_id_b"]),
        })
    out_df = pd.DataFrame(keep, columns=["path_a", "path_b", "label", "vehicle_id_a", "vehicle_id_b"])
    out_df.to_csv(CSV_OUTPUTS[split], index=False)
    pos = int((out_df["label"] == 1).sum())
    neg = int((out_df["label"] == 0).sum())
    print(f"{split}: {len(df)} -> {len(out_df)} (pos={pos}, neg={neg}) -> {CSV_OUTPUTS[split].name}")

## Spot check

Visualise a few before / after crops to make sure the detector did its job.

In [ ]:
import matplotlib.pyplot as plt
import random

sample = random.sample(list(survivors.items()), k=min(4, len(survivors)))
fig, axes = plt.subplots(len(sample), 2, figsize=(8, 3 * len(sample)))
if len(sample) == 1:
    axes = [axes]
for (rel_in, rel_out), (ax_in, ax_out) in zip(sample, axes):
    ax_in.imshow(Image.open(PAIRS_ROOT / rel_in)); ax_in.set_title("before"); ax_in.axis("off")
    ax_out.imshow(Image.open(PAIRS_ROOT / rel_out)); ax_out.set_title("after"); ax_out.axis("off")
plt.tight_layout(); plt.show()

## Next step

In `TrainCosineEncoder.ipynb` change the dataset paths:

```python
TRAIN_CSV = DATA_ROOT / "pairs_train_clean.csv"
VAL_CSV   = DATA_ROOT / "pairs_val_clean.csv"
TEST_CSV  = DATA_ROOT / "pairs_test_clean.csv"
```

`DATA_ROOT` itself stays the same — paths inside the new CSVs already point at `images_cropped/`.